# 10_middleware

对应 `index.md` 第 10 阶段：`create_agent` 的 middleware，以及它取代 `08_agents` 手写循环的哪几件事。

笔记本在 `codes/langchain/10_middleware/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langchain/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

本阶段用 `langchain.agents.create_agent`（`08_agents` 是手写循环，没用它）。模型统一 `init_chat_model("deepseek:...")` 或直接传 `"deepseek:deepseek-chat"`；密钥 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_MODEL`）。

middleware 都从 `langchain.agents.middleware` 导入。


In [1]:
import os
from pathlib import Path


def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "10_middleware" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "10_middleware"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here


def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value


ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))


(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/10_middleware'),
 True)

## 1. 钩子挂在哪里

`create_agent(model, tools, middleware=[...])` 是在 `08_agents` 那套手写循环外面加了一层。先看最简单的两个钩子：`@before_model` 和 `@after_model`。

已有工具 `add`。

1. 写两个钩子函数，各自打印一行（带上当前 `state["messages"]` 的条数），并用 `create_agent` 建一个 agent，`middleware` 传这两个钩子。
2. 用这个 agent `invoke` 一条用户消息：`用工具算 2+3`。
3. 打印每个钩子各触发了多少次，以及最终答案。

回答：这一次 invoke 里模型被调用了几次？为什么不是一次？


In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


# 作答

from langchain.agents.middleware import after_model, before_model
from langchain.chat_models import init_chat_model

before_count = 0
after_count = 0


@before_model
def before_model_hook(state, runtime):
    global before_count
    before_count += 1
    print(f"[before_model] messages: {len(state['messages'])}")
    return None


@after_model
def after_model_hook(state, runtime):
    global after_count
    after_count += 1
    print(f"[after_model] messages: {len(state['messages'])}")
    return None


agent = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash"),
    tools=[add],
    middleware=[before_model_hook, after_model_hook],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})

print(f"before_model 被调用了 {before_count} 次")
print(f"after_model 被调用了 {after_count} 次")
print("最终答案:", response["messages"][-1].content)
#评阅
# 对。两个钩子各 2 次，最终 2 + 3 = 5。
#
# 回答：模型被调用了 2 次。第一次决定调 add，执行完把 ToolMessage 追加回历史后，
# 还要再问一次模型才能生成最终回答。1 次 invoke ≠ 1 次模型调用，这是本阶段的起点。

#参考答案
# before_count = after_count = 0
#
# @before_model
# def before_model_hook(state, runtime):
#     global before_count
#     before_count += 1
#     print("[before_model]", len(state["messages"]))
#     return None
#
# 同理写 after_model_hook；两钩子各会打两行，计数 2 / 2。


[before_model] messages: 1
[after_model] messages: 2
[before_model] messages: 3
[after_model] messages: 4
before_model 被调用了 2 次
after_model 被调用了 2 次
最终答案: 2 + 3 = **5**


## 2. 类式中间件与自定义 state

装饰器只能写无状态的小钩子。要在同一次运行里记东西，用类式 `AgentMiddleware` 加自己的 `state_schema`。

`AgentState` 只有 `messages`、`jump_to`、`structured_response` 三个字段。

1. 定义 `class MyState(AgentState)`，加一个 `calls: NotRequired[int]`。
2. 写一个 `AgentMiddleware` 子类，`state_schema = MyState`，在 `before_model` 里把 `state["calls"]`（第一次还没有，按 0 算）加一写回。
3. 挂上它，`invoke` 同样的问题，打印最终 state 里的 `calls`。

回答：`calls` 是几次？和第 1 题数出来的钩子次数对不对得上？不用 `state_schema` 会怎样？


In [13]:
from typing_extensions import NotRequired

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


# 作答

class MyState(AgentState):
    calls: NotRequired[int] = 0


class MyMiddleware(AgentMiddleware):
    state_schema = MyState

    def before_model(self, state, runtime):
        return {"calls": state.get("calls", 0) + 1}


middleware = MyMiddleware()

agent = create_agent(
    "deepseek:deepseek-v4-flash",
    [add],
    middleware=[middleware],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})

print(f"最终答案: {response['messages'][-1].content}")
print(f"calls: {response['calls']}")
#评阅
# 对，calls 最终是 2，和上一题数出来的钩子次数一致。
#
# 回答：字段活在 state 里，不在 middleware 实例上（所以最后读 response["calls"]，
# 不是 middleware.calls）。不写 state_schema 也能跑，但 return {"calls": ...}
# 会被静默丢弃——不报错，response 里就是不出现这个键，最难查的一类坑。
# 另外字段要 NotRequired 或给默认值，否则第一次进钩子读不到会 KeyError。

#参考答案
# class MyState(AgentState):
#     calls: NotRequired[int] = 0
#
# class MyMiddleware(AgentMiddleware):
#     state_schema = MyState
#
#     def before_model(self, state, runtime):
#         return {"calls": state.get("calls", 0) + 1}
#
# 建 agent 时 middleware=[MyMiddleware()]，最后 print(response["calls"])。


最终答案: 2 + 3 = **5**
calls: 2


## 3. 改写进模型的请求

`@wrap_model_call` 能拿到这一次要发给模型的完整请求（`ModelRequest`），改完再交给 `handler(request)` 放行。

1. 写一个 `@wrap_model_call`，先打印 `request` 的字段名和 `request.system_message`。
2. 用 `request.override(system_prompt=...)` 注入一条系统指令，要求回答结尾都补一句「已核对」。
3. 建 agent 并 `invoke`：`用工具算 2+3`，确认模型照做了。

回答：`request.messages` 和 `request.system_message` 有什么区别？你改的是哪一个？


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


# 作答

from rich import print as rprint

@wrap_model_call
def inject(request, handler):
    print(type(request))
    rprint(request)
    print(type(handler))
    rprint(handler)
    return handler(request.override(system_prompt="回答结尾补一句，[已核对]"))

from langchain_ollama import ChatOllama

agent = create_agent(
    ChatOllama(model="deepseek-r1:7b"),
    [add],
    middleware=[inject],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})

print(response["messages"][-1].content)
#评阅
# 能跑通，但有三处没按题目走：
# 1) 题目要打印 request 的字段名和 request.system_message，你只打了 type(request)；
#    rprint(request) 在这个笔记本里没产生输出（rich 走的不是 stdout），等于没看到内容。
# 2) 模型换成了本地 ChatOllama(deepseek-r1:7b)。它不出 tool_calls，所以这次
#    "2 + 3 = 5" 是模型自己算的，[已核对] 加上了但工具没被调用；
#    题目想验证的是真模型 + 工具这条路径。
# 3) 没回答 request.messages 和 request.system_message 的区别。
#
# 回答：messages 是这次要发的完整对话历史；system_message 是单独的系统消息，
# 默认是 None，只有 create_agent(..., system_prompt=...) 之后才有值。
# request.override(system_prompt=...) 改的是后者，不是往 messages 里塞。

#参考答案
# @wrap_model_call
# def inject(request, handler):
#     print("字段:", list(request.__dataclass_fields__))
#     print("system_message:", request.system_message)
#     return handler(request.override(system_prompt="回答结尾补一句「已核对」"))
#
# agent = create_agent("deepseek:deepseek-chat", tools=[add], middleware=[inject])
# print(agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})["messages"][-1].content)


<class 'langchain.agents.middleware.types.ModelRequest'>


ModelRequest(
    model=ChatOllama(
        metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}},
        model='deepseek-r1:7b'
    ),
    messages=[
        HumanMessage(
            content='用工具算 2+3',
            additional_kwargs={},
            response_metadata={},
            id='e0681f23-3e5e-47f5-a74c-dfee2b5590a4'
        )
    ],
    system_message=None,
    tool_choice=None,
    tools=[
        StructuredTool(
            name='add',
            description='两数相加。',
            args_schema=<class 'langchain_core.utils.pydantic.add'>,
            func=<function add at 0x11fd4c880>
        )
    ],
    response_format=None,
    state={
        'messages': [
            HumanMessage(
                content='用工具算 2+3',
                additional_kwargs={},
                response_metadata={},
                id='e0681f23-3e5e-47f5-a74c-dfee2b5590a4'
            )
        ]
    },
    runtime=Runtime(
        context=None,
        store=None,
        stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x11f74f480>,
        heartbeat=<function _no_op_heartbeat at 0x10e71a610>,
        previous=None,
        execution_info=ExecutionInfo(
            checkpoint_id='1f1bad3c-de53-6b44-8000-77a7365e41d0',
            checkpoint_ns='model:404b63e3-0e67-595d-2e0d-68485b9a8aa7',
            task_id='404b63e3-0e67-595d-2e0d-68485b9a8aa7',
            thread_id=None,
            run_id=None,
            node_attempt=1,
            node_first_attempt_time=1790555428.041462
        ),
        server_info=None,
        control=<langgraph.runtime.RunControl object at 0x11fd110f0>
    ),
    model_settings={}
)

<class 'function'>


<function create_agent.<locals>._execute_model_sync at 0x11fd35bc0>



2 + 3 = 5

[已核对]


## 4. 工具失败时的兜底

`@wrap_tool_call` 包住工具执行，`request.tool_call` 和 `request.tool` 都能读。

已有会抛错的工具 `boom`。

1. 写一个 `@wrap_tool_call`：`try` 里 `return handler(request)`，`except` 时返回一条 `ToolMessage`（`content` 写清失败原因，`tool_call_id` 用 `request.tool_call["id"]`）。
2. 挂上它，`invoke`：`调用 boom 算 1`，确认模型收到的是兜底消息而不是异常。
3. 再把 middleware 换成内置的 `ToolRetryMiddleware(max_retries=2, initial_delay=0.1)`，跑同一个问题。

回答：这两种兜底的区别是什么？


In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware, wrap_tool_call
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool


@tool
def boom(x: int) -> int:
    """一个必定失败的工具。"""
    raise RuntimeError("故意失败")


# 作答

from rich import print as rprint
from langchain_ollama import ChatOllama

@wrap_tool_call
def catch(request, handler):
    try:
        return handler(request)
    except Exception as e:
        rprint(e)
        return ToolMessage(
            content=f"工具执行失败：{e}",
            tool_call_id=request.tool_call["id"],
        )

agent = create_agent(
    # ChatOllama(model="deepseek-r1:7b"),
    init_chat_model("deepseek:deepseek-v4-flash"),
    [boom],
    middleware=[catch],
)

response = agent.invoke({"messages": [HumanMessage("调用 boom 算 1")]})

for msg in response["messages"]:
    msg.pretty_print()

agent = create_agent(
    init_chat_model("deepseek:deepseek-v4-flash"),
    [boom],
    middleware=[ToolRetryMiddleware(max_retries=2, initial_delay=0.1)],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})

for msg in response["messages"]:
    msg.pretty_print()


#评阅
# 自定义兜底那段对：异常被吞掉，模型收到的是 ToolMessage("工具执行失败：故意失败")，
# 整个 invoke 没崩。ToolRetryMiddleware 也挂对了。
#
# 三处要改：
# 1) 第二段仍用 "用工具算 2+3"，但这次只挂了 boom，题面要的是同一个 "调用 boom 算 1"，
#    否则模型面对一个只会失败的工具有点为难，问题也没对准。
# 2) 没回答两种兜底的区别。
# 3) rprint(e) 没输出（同 rich 的问题），用 print(e) 才看得到。
#
# 回答：wrap_tool_call 是"你自己决定怎么办"——吞掉、换结果、改成别的调用都行；
# ToolRetryMiddleware 是固定的"重试 N 次再放弃"，且失败最终还是以异常形式冒到上层。

#参考答案
# @wrap_tool_call
# def catch(request, handler):
#     try:
#         return handler(request)
#     except Exception as e:
#         return ToolMessage(
#             content=f"工具执行失败：{e}",
#             tool_call_id=request.tool_call["id"],
#         )
#
# 再换成 ToolRetryMiddleware(max_retries=2, initial_delay=0.1)，
# 同一个问题各跑一次，对比返回的消息序列。


故意失败

================================ Human Message =================================

调用 boom 算 1
================================== Ai Message ==================================
Tool Calls:
  boom (call_00_j3NwU9Uimt0yXKFDxuoe6698)
 Call ID: call_00_j3NwU9Uimt0yXKFDxuoe6698
  Args:
    x: 1
================================= Tool Message =================================

工具执行失败：故意失败
================================== Ai Message ==================================

如你所料，工具执行失败了。

**调用结果：** `boom(x=1)` → 工具执行失败：故意失败

这个 `boom` 工具被定义为「必定失败的工具」，所以无论传入什么参数都会返回失败。这不是参数写错了（`1` 是合法的整数输入），而是工具本身的行为如此。

如果你有实际想完成的任务，告诉我目标，我可以换用其他方式帮你处理。
================================ Human Message =================================

用工具算 2+3
================================== Ai Message ==================================
Tool Calls:
  boom (call_00_t529NE1jMvgWeyoMclaq3117)
 Call ID: call_00_t529NE1jMvgWeyoMclaq3117
  Args:
    x: 5
================================= Tool Message =================================
Na

## 5. 给 agent 装上限

`08_agents` 第 4 题是自己写 `max_steps`，现在用内置的。

1. 挂 `ModelCallLimitMiddleware(run_limit=1)`，跑一个需要工具往返的问题（`用工具算 2+3`），打印最后一条消息。
2. 换成 `ToolCallLimitMiddleware(tool_name="add", run_limit=0, exit_behavior="continue")`，打印返回的消息序列。
3. 把两个的 `exit_behavior` 都改成 `"error"`，各跑一次，打印异常类型。

回答：两个上限分别卡在哪一步？`"continue"` 和 `"error"` 的差别是什么？


In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


# 作答

agent = create_agent(
    model = init_chat_model("deepseek:deepseek-v4-flash"),
    tools=[add],
    middleware=[ModelCallLimitMiddleware(run_limit=1)],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})

for msg in response["messages"]:
    msg.pretty_print()

agent = create_agent(
    init_chat_model("deepseek:deepseek-v4-flash"),
    [add],
    middleware=[ToolCallLimitMiddleware(tool_name="add", run_limit=0, exit_behavior="continue")],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})

for msg in response["messages"]:
    msg.pretty_print()


agent = create_agent(
    init_chat_model("deepseek:deepseek-v4-flash"),
    [add],
    middleware=[ToolCallLimitMiddleware(tool_name="add", run_limit=0, exit_behavior="error")],
)

try:
    response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})
except Exception as e:
    print(type(e).__name__, e)
#评阅
# 前两段对：
#   ModelCallLimitMiddleware(run_limit=1) -> 最后一条是 "Model call limits exceeded: run limit (1/1)"
#   ToolCallLimitMiddleware(continue)     -> 工具调用被换成提示 ToolMessage，模型自己算出了 5
#   ToolCallLimitMiddleware(error)        -> ToolCallLimitExceededError
#
# 缺一步：ModelCallLimitMiddleware 的 exit_behavior="error" 没试。
# 这一档抛的是 ModelCallLimitExceededError，和 continue 的形状完全不同，值得补上。
#
# 回答：ModelCallLimit 卡在"调模型"这一步（一次 run 里模型最多被调几次）；
# ToolCallLimit 卡在"执行工具"这一步。continue 把限制信息当成正常结果让流程继续，
# error 直接抛异常结束——前者适合体面降级，后者适合宁可失败也不要错误答案。

#参考答案
# for behavior in ("end", "error"):
#     a = create_agent(
#         "deepseek:deepseek-chat",
#         tools=[add],
#         middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior=behavior)],
#     )
#     try:
#         r = a.invoke({"messages": [HumanMessage("用工具算 2+3")]})
#         print(behavior, "->", r["messages"][-1].content)
#     except Exception as e:
#         print(behavior, "->", type(e).__name__, e)


================================ Human Message =================================

用工具算 2+3
================================== Ai Message ==================================
Tool Calls:
  add (call_00_pjZgAgCZqCZhf0ajZWoF1349)
 Call ID: call_00_pjZgAgCZqCZhf0ajZWoF1349
  Args:
    a: 2
    b: 3
================================= Tool Message =================================
Name: add

5
================================== Ai Message ==================================

Model call limits exceeded: run limit (1/1)
================================ Human Message =================================

用工具算 2+3
================================== Ai Message ==================================

我来调用工具计算。
Tool Calls:
  add (call_00_I2WyiAtsWN3ldoaCkvWx5855)
 Call ID: call_00_I2WyiAtsWN3ldoaCkvWx5855
  Args:
    a: 2
    b: 3
================================= Tool Message =================================
Name: add

Tool call limit exceeded. Do not call 'add' again.
================================== Ai 

## 6. 历史太长自动摘要

`SummarizationMiddleware` 在历史超过阈值时，把前面的对话压成一段摘要，只留最近几条。

1. 用 `trigger=("messages", 4)`、`keep=("messages", 2)` 建一个 `SummarizationMiddleware`，摘要模型也用 `"deepseek:deepseek-chat"`。
2. 再写一个 `@wrap_model_call`，每次打印真正进模型的消息条数、以及第一条消息的内容开头。
3. 建 agent（两个 middleware 都挂上），一次 `invoke` 传这些消息：连续 5 条 `第 i 句：给我记住数字 {i*10}。`，末尾再加一条 `用工具算 2+3`。

回答：轮到第 6 条时，模型看到的是几条消息？原来那 5 句去哪了？最终 state 里的消息条数和进模型的一样吗？


In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, wrap_model_call
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


# 作答

summary = SummarizationMiddleware(
    trigger=("messages", 4), 
    keep=("messages", 2),
    model="deepseek:deepseek-v4-flash")

@wrap_model_call
def call_hanlder(request, handler):
    print(f"真正进模型的消息条数：{len(request.messages)}")
    print(f"第一条消息的内容开头：{request.messages[0].content}")
    return handler(request)

agent = create_agent(
    init_chat_model("deepseek:deepseek-v4-flash"),
    [add],
    middleware=[summary, call_hanlder],
)

messages = [
    HumanMessage(f"第 {i} 句：给我记住数字 {i*10}。")
    for i in range(5)
]
messages.append(HumanMessage("用工具算 2+3"))
response = agent.invoke({"messages": messages})

for msg in response["messages"]:
    msg.pretty_print()


#评阅
# 对。两次进模型都只有 3 条，第一条是 "Here is a summary of the conversation to date:"，
# 后面跟着被保留的最近两条；模型仍然报得出第 0~4 句的数字，说明摘要把前 4 句带过去了。
# 小问题：名字拼错了（call_hanlder -> call_handler），不影响运行。
#
# 回答：前面那 5 句被压成了一段摘要（合成的一条 HumanMessage），只留最近 2 条原文。
# 最终 state 里是 4 条（摘要 + 保留的 AI/Tool + 最后的 AI），和进模型的 3 条不是一回事——
# 摘要发生在每次"调用模型之前"，不写回 state，所以每次都会重新算一遍。

#参考答案
# summary = SummarizationMiddleware(
#     model="deepseek:deepseek-chat",
#     trigger=("messages", 4),
#     keep=("messages", 2),
# )
#
# @wrap_model_call
# def spy(request, handler):
#     print("进模型:", len(request.messages), "| 第一条:", str(request.messages[0].content)[:40])
#     return handler(request)
#
# agent = create_agent("deepseek:deepseek-chat", tools=[add], middleware=[summary, spy])


真正进模型的消息条数：3
第一条消息的内容开头：Here is a summary of the conversation to date:

## SESSION INTENT

The user is issuing a series of numbered instructions asking the assistant to remember specific numbers in sequence. This appears to be a sequential memory task where the user provides statements labeled "第 N 句" (Sentence N) each containing a number for the assistant to retain. The overall goal is to accurately remember and recall these numbers associated with their sentence indices.

## SUMMARY

The user has provided a sequence of memory instructions, each associating a sentence number with a value to remember:

- 第 0 句 (Sentence 0): remember the number **0**
- 第 1 句 (Sentence 1): remember the number **10**
- 第 2 句 (Sentence 2): remember the number **20**
- 第 3 句 (Sentence 3): remember the number **30**

Pattern observed: Each sentence number N is associated with the value N × 10 (i.e., 0→0, 1→10, 2→20, 3→30). This pattern should be noted in case the user continues with additional sentences (e.g

## 7. 工具审批

`HumanInTheLoopMiddleware(interrupt_on={"add": True})` 在工具执行前停下来等人。它需要 `checkpointer` 和 `thread_id`。

1. 建 agent：挂上这个 middleware，`checkpointer=MemorySaver()`，`invoke` 时传 `config={"configurable": {"thread_id": "..."}}`。打印结果里的 `__interrupt__`，看 `action_requests` 和 `review_configs` 各是什么。
2. 用 `Command(resume={"decisions": [{"type": "approve"}]})` 继续，打印最终答案。
3. 换一个 `thread_id` 重跑一遍，decisions 改成 `{"type": "reject"}`，看模型怎么处理。

`decisions` 是列表，每项形如 `{"type": "approve"}`；允许哪些取值，可以从中断负载的 `review_configs` 里读。

回答：这和你练过的 LangGraph `interrupt_before` / `interrupt(payload)` 是什么关系？


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command


@tool
def add(a: int, b: int) -> int:
    """两数相加。"""
    return a + b


# 作答

thread = {"configurable": {"thread_id": "..."}}

agent = create_agent(
    init_chat_model("deepseek:deepseek-v4-flash"),
    [add],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"add": True})],
    checkpointer=MemorySaver(),
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]}, config=thread)

for msg in response["messages"]:
    msg.pretty_print()

from rich import print as rprint

rprint(response["__interrupt__"])

response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=thread,
)

for msg in response["messages"]:
    msg.pretty_print()
#评阅
# 中断和 approve 都跑通了：第一轮停在 add 之前（messages 里只有 Human 和那条待审的 AI），
# resume 之后工具执行、拿到 2 + 3 = 5。
#
# 两项没做完：
# 1) 中断负载等于没看 —— rprint(response["__interrupt__"]) 又没输出（rich 的问题）。
#    结构是 value 里有 action_requests（每项 name / args / description）
#    和 review_configs（每项 action_name / allowed_decisions）。
# 2) 第 3 步的 reject 没做：换个 thread_id 重跑，decisions 用 {"type": "reject"}。
#    decisions 的长度必须等于挂起的工具调用数，给多给少都会 ValueError。
#
# 回答：这就是 LangGraph 的中断，底层还是 interrupt(payload) + Command(resume=...)；
# 你在 04_interrupts_hitl 学的那套语义没变，这里只是把"工具执行前停下等人"封装成了中间件。

#参考答案
# agent = create_agent(
#     "deepseek:deepseek-chat",
#     tools=[add],
#     middleware=[HumanInTheLoopMiddleware(interrupt_on={"add": True})],
#     checkpointer=MemorySaver(),
# )
# cfg = {"configurable": {"thread_id": "t1"}}
# r1 = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]}, config=cfg)
# print(r1["__interrupt__"][0].value)          # action_requests + review_configs
# r2 = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=cfg)
# print(r2["messages"][-1].content)


================================ Human Message =================================

用工具算 2+3
================================== Ai Message ==================================

I'll use the add tool to calculate 2+3.
Tool Calls:
  add (call_00_qcuvmpO1DZvfJjOsd1gT4962)
 Call ID: call_00_qcuvmpO1DZvfJjOsd1gT4962
  Args:
    a: 2
    b: 3


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'add',
                    'args': {'a': 2, 'b': 3},
                    'description': "Tool execution requires approval\n\nTool: add\nArgs: {'a': 2, 'b': 3}"
                }
            ],
            'review_configs': [
                {'action_name': 'add', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}
            ]
        },
        id='997a7e2f21453a9055da874f88fdbd5c',
        response_schema=None
    )
]

================================ Human Message =================================

用工具算 2+3
================================== Ai Message ==================================

I'll use the add tool to calculate 2+3.
Tool Calls:
  add (call_00_qcuvmpO1DZvfJjOsd1gT4962)
 Call ID: call_00_qcuvmpO1DZvfJjOsd1gT4962
  Args:
    a: 2
    b: 3
================================= Tool Message =================================
Name: add

5
================================== Ai Message ==================================

2 + 3 = **5**


## 8. 多个中间件的顺序

1. 写两个 `@wrap_model_call`，分别用 `name="A"` 和 `name="B"`，各自在调用 `handler(request)` 之前和之后打印一行。
2. 传给 `middleware=[A, B]`，跑一次不需要工具的 invoke，打印顺序。

回答：顺序是 `A → B → A` 还是 `A → B → B → A`？为什么是这个形状？


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import HumanMessage


# 作答

@wrap_model_call
def hook1(request, handler):
    print("[A]")
    res = handler(request)
    print("[A] done")
    return res

@wrap_model_call
def hook2(request, handler):
    print("[B]")
    res = handler(request)
    print("[B] done")
    return res

agent = create_agent(
    model = init_chat_model("deepseek:deepseek-v4-flash"),
    tools=[],
    middleware=[hook1, hook2],
)

response = agent.invoke({"messages": [HumanMessage("用工具算 2+3")]})
#评阅
# 只跑到一半：输出停在 [A] [B]，两个 done 都没打出来，顺序也没结论。
# 另外 tools=[] 但问题还是"用工具算 2+3"，题面要的是不需要工具的 invoke。
#
# 回答：A 进 → B 进 → B 出 → A 出（洋葱）。list 里越靠前的 middleware 越在外层，
# 进的时候按顺序，出的时候反过来。而且一次 invoke 里模型可能被调多次，
# 所以完整输出是这四行重复两遍（有工具往返时）。

#参考答案
# @wrap_model_call(name="A")
# def hook_a(request, handler):
#     print("A:enter")
#     resp = handler(request)
#     print("A:exit")
#     return resp
#
# @wrap_model_call(name="B")
# def hook_b(request, handler):
#     print("B:enter")
#     resp = handler(request)
#     print("B:exit")
#     return resp
#
# agent = create_agent("deepseek:deepseek-chat", middleware=[hook_a, hook_b])
# agent.invoke({"messages": [HumanMessage("直接回答：1+1 等于几？不要用工具")]})


[A]
[B]
